# PIDA-RL Diabetes — 01. Fuentes y carga controlada

Este notebook localiza, valida y carga las fuentes públicas de ENSANUT utilizadas para el proyecto. **No crea la cohorte, no imputa valores, no entrena modelos y no modifica los archivos fuente.**

Los microdatos se mantienen fuera del repositorio Git. La trazabilidad de las fuentes se guarda como metadatos ligeros en Google Drive.

## 1. Inicialización de rutas

Esta celda es autosuficiente: reutiliza Drive si ya está montado y reconoce tanto `/content/drive` como `/content/gdrive`.

In [3]:
from pathlib import Path
import os
import sys
import subprocess
import json
from datetime import datetime, timezone

PROJECT_NAME = "pida-rl-datos-publicos"
REPO_URL = "https://github.com/JssSalas/pida-rl-datos-publicos.git"
PROJECT_ROOT = Path("/content") / PROJECT_NAME

# Reutiliza Google Drive si ya está montado.
DRIVE_ROOT = next(
    (
        path
        for path in [
            Path("/content/drive"),
            Path("/content/gdrive")
        ]
        if (path / "MyDrive").is_dir()
    ),
    None
)

# Monta Drive solo si aún no está disponible.
if DRIVE_ROOT is None:
    from google.colab import drive

    preferred_mount = Path("/content/drive")
    alternative_mount = Path("/content/gdrive")

    if (
        preferred_mount.exists()
        and any(preferred_mount.iterdir())
    ):
        mount_point = alternative_mount
    else:
        mount_point = preferred_mount

    if (
        mount_point.exists()
        and any(mount_point.iterdir())
    ):
        raise RuntimeError(
            f"No es posible montar Drive en {mount_point} "
            "porque la carpeta contiene archivos locales. "
            "Reinicia la sesión de Colab o usa un punto de montaje vacío."
        )

    drive.mount(
        str(mount_point),
        force_remount=False
    )

    DRIVE_ROOT = mount_point

MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"
PERSISTENT_ROOT = (
    MYDRIVE_ROOT / "PIDA-RL-Diabetes"
)

DATA_ROOT = PERSISTENT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
INTERIM_DATA_DIR = DATA_ROOT / "interim"
METADATA_DIR = DATA_ROOT / "metadata"

for directory in [
    RAW_DATA_DIR,
    INTERIM_DATA_DIR,
    METADATA_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# Recupera el repositorio temporal si Colab lo eliminó.
if not PROJECT_ROOT.exists():
    print("Repositorio local no encontrado.")
    print("Clonando repositorio...")

    subprocess.check_call([
        "git",
        "clone",
        REPO_URL,
        str(PROJECT_ROOT)
    ])

elif not (PROJECT_ROOT / ".git").is_dir():
    raise RuntimeError(
        f"La ruta existe, pero no parece un repositorio Git: "
        f"{PROJECT_ROOT}"
    )

os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_ROOT)
    )

print("Proyecto:", PROJECT_ROOT)
print("Drive:", DRIVE_ROOT)
print("Datos crudos:", RAW_DATA_DIR)
print("Directorio actual:", Path.cwd())


Mounted at /content/drive
Repositorio local no encontrado.
Clonando repositorio...
Proyecto: /content/pida-rl-datos-publicos
Drive: /content/drive
Datos crudos: /content/drive/MyDrive/PIDA-RL-Diabetes/data/raw
Directorio actual: /content/pida-rl-datos-publicos


## 2. Dependencias de lectura

`pyreadstat` permite leer archivos SPSS `.sav` junto con sus metadatos.

In [4]:
import subprocess

try:
    import pyreadstat
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pyreadstat'])
    import pyreadstat

import pandas as pd

print('pyreadstat:', pyreadstat.__version__)
print('pandas:', pd.__version__)


pyreadstat: 1.3.6
pandas: 2.2.3


## 3. Localizar fuentes

Coloca los archivos ENSANUT preferentemente en `MyDrive/PIDA-RL-Diabetes/data/raw/`. Se soportan nombres alternativos y se hace una búsqueda controlada en `data/raw` e `data/interim`; no se explora todo Drive para evitar tomar archivos equivocados.

In [7]:
SOURCE_SPECS = {
    'adultos': {
        'expected_names': ['AdultosENSANUT2022.sav', 'Adultos_ENSANUT_2022.sav'],
        'keywords': ['adult', 'ensanut', '2022'],
        'required': True
    },
    'antro_tension': {
        'expected_names': ['AntroTensionENSANUT2022.sav', 'Antro_Tension_ENSANUT_2022.sav'],
        'keywords': ['antro', 'tension'],
        'required': False
    },
    'sangre': {
        'expected_names': ['MuestrasSangreENSANUT2022.sav', 'Muestras_Sangre_ENSANUT_2022.sav'],
        'keywords': ['sangre'],
        'required': False
    }
}

SEARCH_DIRS = [RAW_DATA_DIR, INTERIM_DATA_DIR]
ALL_SAV_FILES = sorted({
    path for directory in SEARCH_DIRS if directory.exists()
    for path in directory.rglob('*.sav')
    if path.is_file()
})

def locate_source(spec):
    for expected_name in spec['expected_names']:
        matches = [path for path in ALL_SAV_FILES if path.name.lower() == expected_name.lower()]
        if matches:
            return matches[0]

    keywords = [keyword.lower() for keyword in spec['keywords']]
    matches = [
        path for path in ALL_SAV_FILES
        if all(keyword in path.name.lower() for keyword in keywords)
    ]
    return matches[0] if len(matches) == 1 else None

source_paths = {name: locate_source(spec) for name, spec in SOURCE_SPECS.items()}

print('Archivos .sav detectados:')
for path in ALL_SAV_FILES:
    print(' -', path)

print('\nResolución de fuentes:')
for name, path in source_paths.items():
    status = path if path is not None else 'NO ENCONTRADO'
    print(f' - {name}: {status}')

missing_required = [
    name for name, spec in SOURCE_SPECS.items()
    if spec['required'] and source_paths[name] is None
]

if missing_required:
    raise FileNotFoundError(
        'No se encontraron fuentes obligatorias: ' + ', '.join(missing_required) + '.\n'
        f'Coloca el archivo en: {RAW_DATA_DIR}'
    )


Archivos .sav detectados:
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/raw/Adultos_ENSANUT_2022.sav
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/raw/Antro_Tension_ENSANUT_2022.sav
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/raw/Muestras_Sangre_ENSANUT_2022.sav

Resolución de fuentes:
 - adultos: /content/drive/MyDrive/PIDA-RL-Diabetes/data/raw/Adultos_ENSANUT_2022.sav
 - antro_tension: /content/drive/MyDrive/PIDA-RL-Diabetes/data/raw/Antro_Tension_ENSANUT_2022.sav
 - sangre: /content/drive/MyDrive/PIDA-RL-Diabetes/data/raw/Muestras_Sangre_ENSANUT_2022.sav


## 4. Cargar archivos y metadatos

La carga se realiza en memoria. La cohorte se construirá hasta el notebook 03, después de revisar calidad y diccionario de variables.

In [8]:
frames = {}
metas = {}

for name, path in source_paths.items():
    if path is None:
        print(f'{name}: se omite porque no fue localizado.')
        continue

    df, meta = pyreadstat.read_sav(path, apply_value_formats=False)
    frames[name] = df
    metas[name] = meta
    print(f'{name}: {df.shape[0]:,} filas x {df.shape[1]:,} columnas')

df_adultos = frames['adultos']

# Alias de compatibilidad para los notebooks 02 y 03.
df_diabetes = df_adultos

print('\ndf_adultos y df_diabetes disponibles en memoria.')


adultos: 11,913 filas x 572 columnas
antro_tension: 15,073 filas x 99 columnas
sangre: 7,737 filas x 46 columnas

df_adultos y df_diabetes disponibles en memoria.


## 5. Registro de trazabilidad

Se guarda un inventario de archivos, dimensiones y fecha de carga. No se guardan microdatos ni valores personales en este registro.

In [9]:
source_inventory = []
for name, path in source_paths.items():
    item = {
        'source_key': name,
        'path': str(path) if path else None,
        'filename': path.name if path else None,
        'exists': bool(path and path.is_file()),
        'size_bytes': path.stat().st_size if path and path.is_file() else None,
        'rows': int(frames[name].shape[0]) if name in frames else None,
        'columns': int(frames[name].shape[1]) if name in frames else None,
        'loaded_at_utc': datetime.now(timezone.utc).isoformat()
    }
    source_inventory.append(item)

inventory_path = METADATA_DIR / 'source_inventory.json'
with inventory_path.open('w', encoding='utf-8') as file:
    json.dump(source_inventory, file, ensure_ascii=False, indent=2)

display(pd.DataFrame(source_inventory))
print('Inventario guardado en:', inventory_path)


,source_key,path,filename,exists,size_bytes,rows,columns,loaded_at_utc
0,adultos,/content/drive/MyDrive/PIDA-RL-Diabetes/data/r...,Adultos_ENSANUT_2022.sav,True,14975827,11913,572,2026-09-06T17:46:42.786195+00:00
1,antro_tension,/content/drive/MyDrive/PIDA-RL-Diabetes/data/r...,Antro_Tension_ENSANUT_2022.sav,True,7281131,15073,99,2026-09-06T17:46:42.786918+00:00
2,sangre,/content/drive/MyDrive/PIDA-RL-Diabetes/data/r...,Muestras_Sangre_ENSANUT_2022.sav,True,2268035,7737,46,2026-09-06T17:46:42.787600+00:00


Inventario guardado en: /content/drive/MyDrive/PIDA-RL-Diabetes/data/metadata/source_inventory.json


## 6. Inspección inicial

Esta inspección sirve para confirmar que los archivos se leyeron correctamente. La selección definitiva de variables y la revisión de faltantes ocurren en los notebooks 02 y 03.

In [10]:
for name, df in frames.items():
    print('\n' + '=' * 80)
    print(name.upper())
    print('Dimensiones:', df.shape)
    print('Primeras columnas:', df.columns.tolist()[:25])
    display(df.head(3))



ADULTOS
Dimensiones: (11913, 572)
Primeras columnas: ['FOLIO_I', 'FOLIO_INT', 't_hora', 't_min', 't_sumai', 't_sumaf', 'hora_ini_1', 'fecha_ini_1', 'hora_fin_1', 'fecha_fin_1', 'tiempo1', 'resultado_1', 'hora_ini_2', 'fecha_ini_2', 'hora_fin_2', 'fecha_fin_2', 'tiempo2', 'resultado_2', 'hora_ini_3', 'fecha_ini_3', 'hora_fin_3', 'fecha_fin_3', 'tiempo3', 'resultado_3', 'hora_ini_4']


,FOLIO_I,FOLIO_INT,t_hora,t_min,t_sumai,t_sumaf,hora_ini_1,fecha_ini_1,hora_fin_1,fecha_fin_1,...,tiempo,hora_fin,fecha_fin,completa,otroent,ponde_f,estrato,est_sel,upm,region
0,2022_01001001,2022_01001001_01,19.0,12.0,1140.0,1152.0,19:00:11,14/10/2022,19:12:59,14/10/2022,...,12.0,19:12:59,14/10/2022,1.0,,792.951734,3.0,13.0,0100100011320,4.0
1,2022_01001003,2022_01001003_02,12.0,21.0,725.0,741.0,12:05:36,14/10/2022,12:21:06,14/10/2022,...,16.0,12:21:06,14/10/2022,1.0,,2101.024382,3.0,13.0,0100100011320,4.0
2,2022_01001003,2022_01001003_03,19.0,48.0,1177.0,1188.0,19:37:24,14/10/2022,19:48:42,14/10/2022,...,11.0,19:48:42,14/10/2022,1.0,,1303.313846,3.0,13.0,0100100011320,4.0



ANTRO_TENSION
Dimensiones: (15073, 99)
Primeras columnas: ['FOLIO_I', 'FOLIO_INT', 't_hora', 't_min', 't_sumai', 't_sumaf', 'hora_ini_1', 'fecha_ini_1', 'hora_fin_1', 'fecha_fin_1', 'tiempo1', 'resultado_1', 'hora_ini_2', 'fecha_ini_2', 'hora_fin_2', 'fecha_fin_2', 'tiempo2', 'resultado_2', 'hora_ini_3', 'fecha_ini_3', 'hora_fin_3', 'fecha_fin_3', 'tiempo3', 'resultado_3', 'hora_ini_4']


,FOLIO_I,FOLIO_INT,t_hora,t_min,t_sumai,t_sumaf,hora_ini_1,fecha_ini_1,hora_fin_1,fecha_fin_1,...,comentario,tiempo,hora_fin,fecha_fin,completa,otroent,ponde_f,estrato,est_sel,upm
0,2022_01001001,2022_01001001_01,20.0,31.0,1228.0,1231.0,20:28:39,09/11/2022,20:31:33,09/11/2022,...,,3.0,20:31:33,09/11/2022,1.0,,999.761286,3.0,13.0,0100100011320
1,2022_01001001,2022_01001001_02,20.0,28.0,1226.0,1228.0,20:26:30,09/11/2022,20:28:37,09/11/2022,...,,2.0,20:28:37,09/11/2022,1.0,,1546.785919,3.0,13.0,0100100011320
2,2022_01001003,2022_01001003_02,20.0,36.0,1231.0,1236.0,20:31:35,09/11/2022,20:36:37,09/11/2022,...,,5.0,20:36:37,09/11/2022,1.0,,2926.133592,3.0,13.0,0100100011320



SANGRE
Dimensiones: (7737, 46)
Primeras columnas: ['FOLIO_I', 'FOLIO_INT', 'maquina', 't_hora', 't_min', 't_sumai', 't_sumaf', 'hora_ini_1', 'fecha_ini_1', 'hora_fin_1', 'fecha_fin_1', 'tiempo1', 'resultado_1', 'hora_ini_2', 'fecha_ini_2', 'hora_fin_2', 'fecha_fin_2', 'tiempo2', 'resultado_2', 'hora_ini_3', 'fecha_ini_3', 'hora_fin_3', 'fecha_fin_3', 'tiempo3', 'resultado_3']


,FOLIO_I,FOLIO_INT,maquina,t_hora,t_min,t_sumai,t_sumaf,hora_ini_1,fecha_ini_1,hora_fin_1,...,desc_mun,region,h0302,h0303,Interpretacion_N_PF,Interpretacion_S_PF,ponde_ns_pf,estrato,est_sel,upm
0,2022_01001004,2022_01001004_01,MQ317,13.0,35.0,813.0,815.0,13:33:14,10/11/2022,13:35:20,...,001 AGUASCALIENTES,4.0,2.0,41.0,POSITIVO,POSITIVO,7587.0,3.0,13.0,0100100011320
1,2022_01001009,2022_01001009_02,MQ317,9.0,26.0,561.0,566.0,09:21:48,10/11/2022,09:26:03,...,001 AGUASCALIENTES,4.0,2.0,65.0,POSITIVO,POSITIVO,9499.0,3.0,13.0,0100100011320
2,2022_01001012,2022_01001012_02,MQ317,12.0,5.0,719.0,725.0,11:59:45,09/11/2022,12:05:34,...,001 AGUASCALIENTES,4.0,2.0,68.0,POSITIVO,POSITIVO,6333.0,3.0,13.0,0100100011320


## Resultado esperado

Al terminar deben existir en memoria `df_adultos`, `df_diabetes`, `frames` y `metas`; en Drive debe existir `data/metadata/source_inventory.json`. Continúa con `02_entendimiento_datos.ipynb`.